# C8-embeddings — Review

Work through this notebook *after* the three lesson sessions and
(ideally) the practice sets.
It is a consolidation tool: a summary table, the idiom sheet, a
self-quiz, and pointers to what to redo.
Quiz answers are collapsed at the very end — commit to your answers
before looking.

In [ ]:
import numpy as np

## Concept summary

| Concept | One-line summary | Key fact to retain |
|---|---|---|
| Tokenization | Standardize a string into a token list: `simple_preprocess` lowercases, strips punctuation/digits, keeps 2–15-letter pieces | tokens = occurrences (`len(toks)`, `toks.count(w)`); types = distinct (`len(set(toks))`); `set` loses order *and* counts — ordered dedup is `list(dict.fromkeys(toks))` |
| Word embeddings | Each vocabulary word ↦ a learned dense vector in $\mathbb{R}^{100}$ | similar use → nearby vectors; single coordinates mean nothing; the artifact is fixed (so exact asserts work); gensim stores float32 — cast at the boundary |
| Embedding matrices | Stack $N$ token vectors as rows: $V$ of shape $(N, 100)$; unit-row version named `W` | rows are tokens (the list carries row meaning); normalize via `V / np.sqrt((V*V).sum(axis=1, keepdims=True))` — no `np.linalg`, no loops; C9 consumes `W` as-is |
| Similarity matrices | $S = W W^{\mathsf T}$: all pairwise cosines in one product | $S_{ij} = w_i \cdot w_j = \cos(i,j)$ (entrywise product formula); symmetric; unit diagonal; every entry in $[-1,1]$ (Cauchy–Schwarz) |
| Nearest-neighbor search | Rank a similarity row descending and read off the top $k$ | `np.argsort` is ascending — reverse with `[::-1]`; exclude self by mask `order[order != i]` (the diagonal 1 always ranks first); values calibrate, ranks only order |
| gensim usage | `KeyedVectors`: load once via the shared cache, then membership / lookup / similarity | `w in kv.key_to_index`; `kv[words]` stacks rows in list order; `most_similar(w, topn, restrict_vocab)` excludes the query; gensim computes in float32 — compare with `atol=1e-5` |

## Formula and idiom sheet

**By hand (the forms the exam's registers expect):**

- Cosine: $\cos(u, v) = \dfrac{u \cdot v}{\lVert u \rVert\,\lVert v \rVert} \in [-1, 1]$;
  for **unit** vectors, $\cos(u, v) = u \cdot v$.
- Similarity matrix: $S_{ij} = (W W^{\mathsf T})_{ij} = \sum_k W_{ik} W_{jk} = w_i \cdot w_j$;
  $S = S^{\mathsf T}$; $S_{ii} = 1$.
- Range proof (unit $u, v$): $0 \le \lVert u - v \rVert^2 = 2 - 2\,u \cdot v
  \Rightarrow u \cdot v \le 1$; and $0 \le \lVert u + v \rVert^2 \Rightarrow u \cdot v \ge -1$.
- Tokens vs types: occurrences vs distinct; dedup that keeps first-occurrence
  order: `dict.fromkeys`.

**NumPy + gensim idioms (banned-API-safe):**

```python
V = np.asarray(kv[words], dtype=np.float64)          # (N,100) rows=tokens, cast
norms = np.sqrt((V * V).sum(axis=1, keepdims=True))  # (N,1) -- keepdims!
W = V / norms                                        # unit rows
S = W @ W.T                                          # (N,N) all cosines
order = np.argsort(S[i])[::-1]                       # descending
top = order[order != i][:k]                          # self-excluded top-k
cands = [t for t in dict.fromkeys(toks)              # ordered, unique,
         if t in kv.key_to_index]                    #   in-vocabulary
q = np.asarray(kv[query], dtype=np.float64)
q = q / np.sqrt((q * q).sum())                       # unit query
sims = W @ q                                         # (N,) query cosines
```

Standing habits: assert unit row norms right after normalizing; assert
symmetry and unit diagonal right after building $S$; print values next
to ranked words; `np.isclose` with `atol=1e-6, rtol=0` inside your own float64
pipeline, `atol=1e-5` against gensim's float32 values; never exact
`==` on floats.

## Self-quiz

Fourteen items, all six concepts covered.
Work by hand (calculator-free), then check against the collapsed
answers at the very end.

1. Apply the tokenizer rules by hand:
   `simple_preprocess("The 2 owls saw 1 fox; the fox ran.")` — write
   the token list, then give `(len(toks), len(set(toks)))`.
2. For that same token list, what does `list(dict.fromkeys(toks))`
   return, and which two pieces of information would `set(toks)` have
   destroyed?
3. State the three governing facts about GloVe-style embeddings, one
   clause each.
4. Under this course's convention, give the shape of the embedding
   matrix for 12 tokens, and say what `V[3]` means. Which object —
   code-wise — remembers that row 3 is `"tide"`?
5. Write the two-line banned-API-safe normalization of `V` (norms,
   then `W`), including the cast-free part you may assume: `V` is
   already float64.
6. `(V * V).sum(axis=1)` vs `(V * V).sum(axis=1, keepdims=True)`:
   give both shapes for `V` of shape `(9, 100)`, and name the
   operation that fails if you use the first where the second belongs.
7. Define $S$ from `W`, give its shape for 9 tokens, and state what
   $S_{27}$ measures.
8. State $S$'s three structural properties (symmetry, diagonal,
   range) and, for the range, name the inequality that proves it.
9. By hand: $u = (1, 0, 1)/\sqrt{2}$ and $v = (1, 1, 0)/\sqrt{2}$ are
   unit vectors. Compute their cosine similarity.
10. `s = np.array([0.3, 1.0, 0.7, 0.5])` is row 1 of an $S$ (query
    index 1). Apply the full neighbor recipe and give the top-2
    indices.
11. Why does the raw descending argsort of any similarity row start
    with the query itself, and what is the position-safe exclusion
    idiom?
12. Write the membership test and the boundary-cast lookup for the
    word `"breakwater"`, exactly as the register demands.
13. Your float64 cosine for a pair is `0.4275581`; `kv.similarity`
    returns `0.42755812`. Bug or expected? Which tolerance does this
    course use for that comparison, and why?
14. Order the five retrieval-pipeline stages, and name the stage whose
    omission produces a `KeyError` in the embed stage.

## What to redo, per weak spot

| If you struggled with… | Redo (practice) | Reread (lesson) |
|---|---|---|
| items 1–2 (tokenizer rules, set semantics) | p01, p06, p15 | Session 1 §§1–2 |
| items 3, 13 (embedding facts, float32 boundary) | p02, p05, p19 | Session 1 §§3–4, Session 3 §3 |
| items 4–6 (matrix convention, keepdims register) | p07, p08, p20 | Session 2 §§1–3 |
| items 7–9 (S, its properties, hand cosines) | p03, p04, p10, p11, p12 | Session 2 §§4–5, §7 |
| items 10–11 (argsort, self-exclusion) | p09, p16 | Session 3 §§1–2 |
| items 12–14 (gensim register, pipeline order) | p05, p13, p18 | Session 1 §§4–5, Session 3 §§3–5 |
| integration under pressure | p13, p14, p17, p18 | — |

---

Scored yourself below ~10/14? Use the redo table above, then retake
the quiz.
At 11+ you are ready for `C9-dimensionality-reduction`, which factors
the very `W` you built here.

## Self-quiz answers

<details><summary><b>Click to reveal (commit to your answers first)</b></summary>

1. `['the', 'owls', 'saw', 'fox', 'the', 'fox', 'ran']` — digits die,
   `The`→`the`. `(7, 5)`.
2. `['the', 'owls', 'saw', 'fox', 'ran']` — first occurrences in
   order. A set would destroy the order (arbitrary iteration) and the
   counts (`the`×2, `fox`×2 collapse to one each).
3. Similar use → nearby vectors; individual coordinates carry no
   meaning (only relative geometry does); the artifact is fixed —
   same vectors every load, so exact asserts are reproducible.
4. Shape `(12, 100)`; `V[3]` is token 3's embedding vector, shape
   `(100,)`. The words *list* remembers — the matrix stores only
   numbers.
5. `norms = np.sqrt((V * V).sum(axis=1, keepdims=True))` then
   `W = V / norms`.
6. `(9,)` and `(9, 1)`. The division `V / norms` fails with the flat
   version (broadcasting cannot align 100 with 9).
7. `S = W @ W.T`, shape `(9, 9)`; $S_{27}$ is the cosine similarity
   between token 2 and token 7.
8. $S = S^{\mathsf T}$; $S_{ii} = 1$; all entries in $[-1, 1]$ — by
   Cauchy–Schwarz ($|u \cdot v| \le \lVert u\rVert\,\lVert v\rVert = 1$
   for unit vectors).
9. $u \cdot v = \frac{1}{2}(1\cdot1 + 0\cdot1 + 1\cdot0) = \frac12$ —
   unit vectors, so the dot product *is* the cosine: $0.5$.
10. Descending order of $(0.3, 1.0, 0.7, 0.5)$: indices
    `[1, 2, 3, 0]`; exclude query 1 → `[2, 3, 0]`; top-2 = `[2, 3]`.
11. Because $S_{ii} = 1$ is the row maximum (every other entry is a
    cosine $\le 1$). Idiom: `order[order != i]`.
12. `"breakwater" in kv.key_to_index` and
    `v = np.asarray(kv["breakwater"], dtype=np.float64)`.
13. Expected: gensim computes in float32 (~7 significant digits of
    agreement). Compare with `np.isclose(..., atol=1e-5, rtol=0)` (zero the
    `1e-5` DEFAULT rtol — it would silently widen every test); `1e-6` is
    reserved for values from your own float64 pipeline.
14. Tokenize → dedup (ordered) → filter → embed → rank. Omitting
    **filter** leaves OOV tokens in the candidate list, and
    `kv[cands]` raises `KeyError`.

</details>